In [ ]:
from core.data import subject_ids, session_ids
import numpy as np
import matplotlib.pyplot as plt

import scienceplots  # noqa: F401

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

In [ ]:
from utils.paths import MODELS_DIR, FIGURES_DIR
import pickle

subj_id = "MR83"
sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]

beta_weights = {}

n_cvs = 5
regions = ["all", "DMS", "DLS"]
epoch_keys = ["full", "choice", "reward", "iti"]

tv = "rewarded"
betas_sess = []

seed = 0
for sess_id in sess_ids:
    no_families = False
    betas = {
        reg: {
            epoch: {strategy: [] for strategy in ["mb", "mf"]} for epoch in epoch_keys
        }
        for reg in regions
    }
    # load the families
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "tributaries"
        / "balance_and_norm"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        continue

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)
    for reg in regions:
        print(">", reg)
        for epoch in epoch_keys:
            print(">>", epoch)
            families_mb = res_dict[reg][epoch]["mb"]["families"]
            families_mf = res_dict[reg][epoch]["mf"]["families"]

            if len(families_mb) == 0:
                no_families = True
                break
            beta_mb_sess = []
            beta_mf_sess = []

            family_mb = families_mb[seed]
            family_mf = families_mf[seed]

            beta_mb = family_mb.mod_taskvar.tv.weight.data[:]
            beta_mf = family_mf.mod_taskvar.tv.weight.data[:]

            tv_idxs = []
            tv_labels = []
            counter = 0
            for tv_ in family_mb.task_vars:
                for val in family_mb.trial_data[tv_].unique():
                    if tv_ == tv:
                        tv_idxs.append(counter)
                        if tv_ == "response":
                            if val == 1:
                                val_str = "left"
                            elif val == -1:
                                val_str = "right"
                        if tv_ == "rewarded":
                            if val == 1:
                                val_str = "correct"
                            elif val == 0:
                                val_str = "incorrect"
                        tv_labels.append(f"{tv_}_{val_str}")
                    counter += 1

            # beta_mb_sess.append([beta_mb[tv_idx] for tv_idx in tv_idxs])
            # beta_mf_sess.append([beta_mf[tv_idx] for tv_idx in tv_idxs])
            betas[reg][epoch]["mb"] = np.array([beta_mb[tv_idx] for tv_idx in tv_idxs])
            betas[reg][epoch]["mf"] = np.array([beta_mf[tv_idx] for tv_idx in tv_idxs])
    if not no_families:
        betas_sess.append(betas)

In [ ]:
subj_id = "MR82"
tv = "response"
save_dir = MODELS_DIR / "fit" / subj_id / "betas" / "balance_and_norm"
save_path = save_dir / f"{tv}.pkl"

with open(save_path, "rb") as f:
    res = pickle.load(f)
    betas_sess = res["betas_sess"]
    tv_labels = res["tv_labels"]

In [ ]:
tv_labels

In [ ]:
import matplotlib.pyplot as plt

num_tvs = np.shape(betas_sess[0]["all"]["choice"]["mb"])[0]

colors_tv = ["#17855D", "#9554BD"]

fig, axes = plt.subplots(ncols=3, nrows=3, figsize=(5, 5))

for i, reg in enumerate(regions):
    for j, epoch in enumerate(epoch_keys):
        ax = axes[j][i]

        for betas in betas_sess:
            betas_mb = betas[reg][epoch]["mb"]
            betas_mf = betas[reg][epoch]["mf"]

            for tv_idx in range(num_tvs):
                ax.scatter(
                    betas_mb[tv_idx],
                    betas_mf[tv_idx],
                    s=0.5,
                    color=colors_tv[tv_idx],
                    alpha=0.5,
                    label=f"{tv_labels[tv_idx]}",
                )

        ax.plot([-2, 2], [-2, 2], linewidth=0.5, color="#000000", linestyle="-")
        ax.axhline(y=0, linewidth=0.5, color="#888888", linestyle="--")
        ax.axvline(x=0, linewidth=0.5, color="#888888", linestyle="--")
        ax.set_xlim([-2, 2])
        ax.set_ylim([-2, 2])
        ax.set_xlabel(r"$\beta$ mb")
        ax.set_ylabel(r"$\beta$ mf")

fig.suptitle(tv)
fig.tight_layout()

fpath_png = FIGURES_DIR / "beta_strategy" / subj_id / f"{tv}_beta_strategy-{seed}.png"
fpath_svg = FIGURES_DIR / "beta_strategy" / subj_id / f"{tv}_beta_strategy-{seed}.svg"
fpath_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fpath_png, dpi=300, bbox_inches="tight")
fig.savefig(fpath_svg, dpi=300, bbox_inches="tight")

In [ ]:
num_tvs = np.shape(betas_sess[0]["all"]["choice"]["mb"])[0]

colors_tv = ["#17855D", "#9554BD"]

fig, axes = plt.subplots(ncols=3, nrows=3, figsize=(5.6, 5), sharex="all", sharey="all")

for i, reg in enumerate(regions):
    for j, epoch in enumerate(epoch_keys):
        ax = axes[j][i]

        betas_mb = []
        betas_mf = []
        for betas in betas_sess:
            for tv_idx in range(num_tvs):
                betas_mb.extend(betas[reg][epoch]["mb"][tv_idx])
                betas_mf.extend(betas[reg][epoch]["mf"][tv_idx])

        im = ax.hist2d(
            betas_mb,
            betas_mf,
            range=[[-2, 2], [-2, 2]],
            bins=50,
            cmap="Blues",
            norm="log",
            vmax=10,
            density=True,
        )

        ax.plot([-2, 2], [-2, 2], linewidth=0.5, color="#414141", linestyle="-")
        ax.axhline(y=0, linewidth=0.5, color="#888888", linestyle="--")
        ax.axvline(x=0, linewidth=0.5, color="#888888", linestyle="--")
        ax.set_xlim([-2, 2])
        ax.set_ylim([-2, 2])
        ax.set_xlabel(r"$\beta$ mb")
        ax.set_ylabel(r"$\beta$ mf")

fig.suptitle(tv)
fig.tight_layout()

fig.subplots_adjust(right=0.8)
cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
fig.colorbar(im[3], cax=cbar_ax, ticks=np.logspace(-1, 1, 3, base=10))

# fpath_png = FIGURES_DIR / "beta_strategy" / subj_id / f"{tv}_beta_strategy-{seed}.png"
# fpath_svg = FIGURES_DIR / "beta_strategy" / subj_id / f"{tv}_beta_strategy-{seed}.svg"
# fpath_png.parent.mkdir(parents=True, exist_ok=True)
# fig.savefig(fpath_png, dpi=300, bbox_inches="tight")
# fig.savefig(fpath_svg, dpi=300, bbox_inches="tight")

In [ ]:
from core.data import colors_strategy

num_tvs = np.shape(betas_sess[0]["all"]["choice"]["mb"])[0]


fig, axes = plt.subplots(
    ncols=3, nrows=3, figsize=(5, 5), sharex="all", sharey="all", tight_layout=True
)

for i, reg in enumerate(regions):
    for j, epoch in enumerate(epoch_keys):
        ax = axes[j][i]

        betas_mb = []
        betas_mf = []
        for betas in betas_sess:
            for tv_idx in range(num_tvs):
                betas_mb.extend(betas[reg][epoch]["mb"][tv_idx])
                betas_mf.extend(betas[reg][epoch]["mf"][tv_idx])

        ax.hist(
            betas_mb,
            bins=np.linspace(-2, 2, 26),
            color=colors_strategy["mb"],
            density=True,
            histtype="step",
            label=r"$\beta$ mb",
        )
        ax.hist(
            betas_mf,
            bins=np.linspace(-2, 2, 26),
            color=colors_strategy["mf"],
            density=True,
            histtype="step",
            label=r"$\beta$ mf",
        )

        ax.legend()
        ax.set_xlim([-2, 2])

fig.suptitle(tv)